# Verify Intelligent Routing

This notebook tests llm-d's routing intelligence in detail — confirming that the Endpoint Picker (EPP) makes cache-aware and load-aware routing decisions rather than round-robin.

**What we'll do:**
1. Check EPP pod discovery and metrics collection
2. Send concurrent requests and observe routing distribution
3. Verify prefix-cache-aware routing (same prompt → same pod)
4. Monitor KV cache utilization across replicas
5. Compare latency: unique prompts vs. cached-prefix prompts

## 1. Check EPP Discovery and Metrics

The EPP must discover all vLLM pods in the InferencePool and scrape their `/metrics` endpoint for real-time KV-cache and queue information.

In [ ]:
%%bash
echo "=== EPP Pod ==="
oc get pods -n llm-d-serving -l app.kubernetes.io/component=scheduler

echo ""
echo "=== InferencePool Members ==="
oc get pods -n llm-d-serving -l app.kubernetes.io/name=qwen3-coder-fp8 -o wide

echo ""
echo "=== EPP Logs (last 20 lines) ==="
EPP_POD=$(oc get pods -n llm-d-serving -l app.kubernetes.io/component=scheduler -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
if [ -n "$EPP_POD" ]; then
  oc logs -n llm-d-serving $EPP_POD --tail=20 2>/dev/null | grep -E "discover|endpoint|metrics"
else
  echo "EPP pod not found — check deployment"
fi

## 2. Concurrent Requests — Routing Distribution

Send multiple requests in parallel and check which pod served each one. With a single replica, all go to one pod. With multiple replicas, EPP distributes based on scoring.

In [ ]:
import subprocess, json, os, time
from concurrent.futures import ThreadPoolExecutor

cluster_domain = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
).stdout.strip()

MODEL_URL = f"https://maas.{cluster_domain}"
MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8"

def send_request(prompt, request_id):
    """Send a single chat completion request and measure timing."""
    import urllib.request, ssl, time
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    
    data = json.dumps({
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 30
    }).encode()
    
    req = urllib.request.Request(
        f"{MODEL_URL}/v1/chat/completions",
        data=data,
        headers={"Content-Type": "application/json"}
    )
    start = time.time()
    resp = urllib.request.urlopen(req, context=ctx)
    elapsed = time.time() - start
    return {"id": request_id, "elapsed": round(elapsed, 2)}

print(f"Sending 6 concurrent requests to {MODEL_URL}...")
prompts = [f"Say the number {i}" for i in range(1, 7)]

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(send_request, p, i) for i, p in enumerate(prompts, 1)]
    results = [f.result() for f in futures]

print("\nResults:")
for r in results:
    print(f"  Request {r['id']}: {r['elapsed']}s")

print("\n→ Check pod metrics to see routing distribution:")
print("  oc exec <vllm-pod> -- curl -s localhost:8000/metrics | grep request_success")

## 3. Prefix-Cache-Aware Routing

The key test: send requests with an **identical system prompt** and verify EPP routes them to the **same pod** (the one with the cached prefix).

In [ ]:
import time

SYSTEM_PROMPT = """You are an expert Python developer working on a FastAPI microservice.
Follow PEP 8, use type hints on all functions, and write Google-style docstrings.
Use dependency injection for database sessions. Handle errors with HTTPException."""

def send_with_system_prompt(user_message, request_id):
    import urllib.request, ssl
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    
    data = json.dumps({
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ],
        "max_tokens": 30
    }).encode()
    
    req = urllib.request.Request(
        f"{MODEL_URL}/v1/chat/completions",
        data=data,
        headers={"Content-Type": "application/json"}
    )
    start = time.time()
    resp = urllib.request.urlopen(req, context=ctx)
    elapsed = time.time() - start
    return {"id": request_id, "elapsed": round(elapsed, 3)}

print("Sending 5 sequential requests with IDENTICAL system prompt...")
print(f"System prompt length: ~{len(SYSTEM_PROMPT.split())} words\n")

results = []
for i in range(1, 6):
    r = send_with_system_prompt(f"Write function #{i}: a utility to validate email addresses.", i)
    results.append(r)
    print(f"  Request {i}: {r['elapsed']}s")

print("\n--- Analysis ---")
first = results[0]['elapsed']
rest_avg = sum(r['elapsed'] for r in results[1:]) / len(results[1:])
improvement = ((first - rest_avg) / first) * 100

print(f"First request (cold prefix): {first}s")
print(f"Avg subsequent (cached prefix): {rest_avg:.3f}s")
if improvement > 0:
    print(f"Improvement: {improvement:.1f}% faster with prefix cache ✅")
else:
    print(f"No improvement detected — prefix cache may need more prompt tokens to show benefit")

## 4. KV Cache Utilization

Check the current KV cache usage on each vLLM pod. Higher utilization means the pod has more active sequences cached — EPP uses this to avoid overloading any single replica.

In [ ]:
%%bash
echo "=== vLLM Pod Metrics ==="
for POD in $(oc get pods -n llm-d-serving -l app.kubernetes.io/name=qwen3-coder-fp8 -o jsonpath='{.items[*].metadata.name}'); do
  echo ""
  echo "Pod: $POD"
  oc exec -n llm-d-serving $POD -- curl -s localhost:8000/metrics 2>/dev/null | grep -E "gpu_cache_usage|num_requests|prefix_cache" | head -10
done

## 5. Latency Comparison: Unique vs. Cached Prompts

Compare TTFT for requests with completely unique prompts (cache miss every time) vs. requests sharing the same system prompt (cache hit after first request).

In [ ]:
import time, statistics

def measure_ttft(system_prompt, user_msg):
    import urllib.request, ssl
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    
    messages = [{"role": "user", "content": user_msg}]
    if system_prompt:
        messages.insert(0, {"role": "system", "content": system_prompt})
    
    data = json.dumps({
        "model": MODEL_NAME,
        "messages": messages,
        "max_tokens": 20,
        "stream": False
    }).encode()
    
    req = urllib.request.Request(
        f"{MODEL_URL}/v1/chat/completions",
        data=data,
        headers={"Content-Type": "application/json"}
    )
    start = time.time()
    urllib.request.urlopen(req, context=ctx)
    return time.time() - start

SHARED_PROMPT = "You are a Python expert. Use type hints. Follow PEP 8. Write clean, testable code." * 5

print("=== Test A: Unique prompts (no prefix sharing) ===")
unique_times = []
for i in range(5):
    unique_prompt = f"Context #{i}: " + "x" * 200
    t = measure_ttft(unique_prompt, f"Write function {i}")
    unique_times.append(t)
    print(f"  Request {i+1}: {t:.3f}s")

print(f"\n  Mean: {statistics.mean(unique_times):.3f}s")

print("\n=== Test B: Shared system prompt (prefix cache) ===")
shared_times = []
for i in range(5):
    t = measure_ttft(SHARED_PROMPT, f"Write function {i}")
    shared_times.append(t)
    print(f"  Request {i+1}: {t:.3f}s")

print(f"\n  Mean: {statistics.mean(shared_times):.3f}s")

print("\n=== Comparison ===")
unique_mean = statistics.mean(unique_times)
shared_mean = statistics.mean(shared_times[1:])  # exclude first (cold)
print(f"  Unique prompts avg:    {unique_mean:.3f}s")
print(f"  Shared prompt avg (warm): {shared_mean:.3f}s")
if shared_mean < unique_mean:
    pct = ((unique_mean - shared_mean) / unique_mean) * 100
    print(f"  Prefix cache benefit: {pct:.1f}% lower latency ✅")

## Summary

| Test | Result |
|------|--------|
| EPP pod discovery | Verified — all vLLM pods visible in InferencePool |
| Concurrent routing | Requests distributed based on EPP scoring |
| Prefix-cache routing | Same system prompt → routed to cached pod |
| KV cache metrics | Available via pod `/metrics` endpoint |
| Latency improvement | Shared-prefix requests faster than unique-prefix |

**Key takeaways:**
- llm-d EPP makes stateful routing decisions — not round-robin
- Coding assistant workloads benefit significantly from prefix caching (shared system prompts)
- Multiple developers sharing the same assistant config will see lower TTFT over time

→ Continue to **Phase 5** (`5_developer_experience/`) to set up Dev Spaces with pre-configured AI extensions that all hit this llm-d endpoint.